1.DATAFRAME CREATION

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("Parquet").getOrCreate()
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000, 0),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000, 0),
(103,"Rahul Sharma","Mumbai","Dermatology",1500, 0),
(104,"Priya Nair","Bangalore","Cardiology",5000, 0),
(105,"Vikram Singh","Chennai","Neurology",7000, 0)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,0
102,Sneha Kapoor,Delhi,Orthopedics,3000,0
103,Rahul Sharma,Mumbai,Dermatology,1500,0
104,Priya Nair,Bangalore,Cardiology,5000,0
105,Vikram Singh,Chennai,Neurology,7000,0


2.PARQUET CREATION

In [0]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

3.READ PARQUET

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
parquet_df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          0|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          0|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          0|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          0|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          0|
+--------+------------+---------+-----------+----------------+-----------+



4.PARQUET SCHEMA

In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)
 |-- tests_count: long (nullable = true)



5.COLUMN PROJECTION

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.select("patient_name","city") \
.show()

+------------+---------+
|patient_name|     city|
+------------+---------+
| Arjun Reddy|Hyderabad|
|  Priya Nair|Bangalore|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
|Sneha Kapoor|    Delhi|
+------------+---------+



6.FILTER DATA

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.filter("consultation_fee > 3000") \
.show()

+--------+------------+---------+----------+----------------+-----------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|
+--------+------------+---------+----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          0|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          0|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          0|
+--------+------------+---------+----------+----------------+-----------+



PARTIONED PARQUET WRITE

In [0]:
df.write \
.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/patient_parquet_partitioned")

READ PARTIONED PARQUET


In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+-----------+---------+
|visit_id|patient_name| department|consultation_fee|tests_count|     city|
+--------+------------+-----------+----------------+-----------+---------+
|     103|Rahul Sharma|Dermatology|            1500|          0|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|          0|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|          0|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|          0|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|          0|Bangalore|
+--------+------------+-----------+----------------+-----------+---------+



PARTIONED PRUNING


In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
.filter("city = 'Hyderabad'") \
.show()

+--------+------------+----------+----------------+-----------+---------+
|visit_id|patient_name|department|consultation_fee|tests_count|     city|
+--------+------------+----------+----------------+-----------+---------+
|     101| Arjun Reddy|Cardiology|            5000|          0|Hyderabad|
+--------+------------+----------+----------------+-----------+---------+



APPEND MODE


In [0]:
columns = ["visit_id","patient_name","city","department","consultation_fee"]

new_data = [
    (106,"Ananya Das","Kolkata","Orthopedics",3000)
]

new_df = spark.createDataFrame(new_data, columns)

new_df.write \
.mode("append") \
.format("delta").save("dbfs:/tmp/patient_parquet")

In [0]:
df.write \
.mode("overwrite") \
.format("delta").save("/tmp/patient_parquet")

In [0]:
%sql
CREATE OR REPLACE VIEW patient_parquet_view AS 
SELECT * FROM delta.`dbfs:/tmp/patient_parquet`;

In [0]:
%sql
SELECT * FROM patient_parquet_view;

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,0
102,Sneha Kapoor,Delhi,Orthopedics,3000,0
103,Rahul Sharma,Mumbai,Dermatology,1500,0
104,Priya Nair,Bangalore,Cardiology,5000,0
105,Vikram Singh,Chennai,Neurology,7000,0


In [0]:
%sql
CONVERT TO DELTA parquet.`dbfs:/tmp/patient_parquet`;

In [0]:
%sql
UPDATE delta.`dbfs:/tmp/patient_parquet`
SET consultation_fee = 6000
WHERE visit_id = 101;

num_affected_rows
1
